# Per-UID LSTM for IEEE-CIS Fraud — PyTorch version

This notebook mirrors `Time_Series_LSTM_per_UID.ipynb` (Keras) but is written in
PyTorch, following the structural pattern from the Kaggle reference:

> [arunmohan003 — *Sentiment analysis using LSTM - PyTorch*](https://www.kaggle.com/code/arunmohan003/sentiment-analysis-using-lstm-pytorch)

Same upstream logic as the Keras notebook:

- Load preprocessed checkpoints (`X_train_copy4.pkl`, `X_test_copy4.pkl`, `y_train.pkl`).
- Bridge train+test before windowing so test rows can use train history.
- Add three time-gap features per UID.
- Standardize (fit on train rows only).
- Build per-UID sliding windows of length `WINDOW`.
- Train under **strict expanding-window time validation** with `MIN_TRAIN_MONTHS = 3`.
- Save OOF and test predictions for ensembling with your XGBoost OOF.

## What's adapted from the sentiment-analysis reference

| arunmohan003's notebook | This notebook |
|--|--|
| Word indices `(batch, seq_len)` → `nn.Embedding` → `(batch, seq_len, embed_dim)` | Numeric features `(batch, seq_len, n_features)` directly into LSTM (no embedding) |
| `vocab_size`, `embedding_dim` hyperparameters | `n_features` only — no vocab |
| `nn.LSTM(input_size=embedding_dim, ...)` | `nn.LSTM(input_size=n_features, ...)` |
| `model.init_hidden(batch_size)` per iteration | Same pattern, kept for fidelity |
| `nn.BCELoss` after sigmoid | Same |
| Manual training loop with `optimizer.zero_grad()`, `loss.backward()`, `clip_grad_norm_`, `optimizer.step()` | Same |
| Best model saved by validation loss | Best model saved by **validation AUC** (better metric for fraud) |

The reason there's no embedding layer: in sentiment analysis each word is a discrete
token that has to be turned into a continuous vector. Your fraud features are already
continuous after `StandardScaler` (and previously-categorical fields like `card1` were
already integer-encoded by the upstream pipeline), so the LSTM can ingest them directly.


## MPS / Apple Silicon GPU notes

PyTorch supports Apple Silicon GPU via the **MPS** (Metal Performance Shaders) backend.
The config cell below picks the best available device automatically: `mps` if you're on
M-series, else `cuda`, else `cpu`.

A few specifics for MPS:

- Some ops fall back to CPU silently. For LSTM this works but you may see warnings
  about unsupported dtypes — mostly harmless.
- `torch.compile(...)` doesn't help much on MPS yet (Metal backend is limited),
  so this notebook doesn't use it.
- Mixed-precision (`autocast`) is supported but not used here — fp32 is more stable
  for LSTM on MPS, and the speed difference is small.
- `num_workers > 0` in `DataLoader` can deadlock on macOS in Jupyter. We use
  `num_workers=0` and rely on the unified-memory architecture for fast host-device
  transfer.


In [1]:
import sys, os
print(sys.executable)
print(os.environ.get("CONDA_DEFAULT_ENV"))

/Users/hovietbach/miniforge3/envs/Financial_Fraud_Detection_Thesis/bin/python
Financial_Fraud_Detection_Thesis


In [2]:
# 0. Imports and config — PyTorch + MPS-aware
import os, gc, math, time, datetime, warnings, copy
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.metrics import roc_auc_score
from sklearn.preprocessing import StandardScaler

import torch
from torch import nn
from torch.utils.data import Dataset, DataLoader

# ----- Configuration -----
DATA_DIR = '/ieee-fraud-detection/pkl_exported_files'

WINDOW              = 20      # was 5  (change F)
MIN_TRAIN_MONTHS    = 3
BATCH               = 1024
EPOCHS              = 30      # was 12 (change I)
LR                  = 1e-3
WEIGHT_DECAY        = 2e-4
GRAD_CLIP           = 1.0     # was 0.5 (change D)
EARLY_STOP_PATIENCE = 6
SEED                = 42

HIDDEN_DIM   = 128
NUM_LAYERS   = 2
DROPOUT      = 0.3
N_SEEDS      = 3              # for seed ensembling (change H)
USE_POS_WEIGHT = False        # plain BCE for AUC (change C)
USE_STATIC_TOWER = False       # dual-tower (change B)

# ----- Reproducibility -----
torch.manual_seed(SEED); np.random.seed(SEED)

# ----- Device selection -----
if torch.backends.mps.is_available():
    device = torch.device('mps')
elif torch.cuda.is_available():
    device = torch.device('cuda')
else:
    device = torch.device('cpu')

print(f'PyTorch {torch.__version__}  device={device}')


PyTorch 2.11.0  device=mps


In [4]:
x = torch.randn(1024, 5, 250).to(device)
print(x.device)

mps:0


## 5. Build per-UID windows on the combined frame

In [ ]:
data = np.load("/ieee-fraud-detection/split_data.npz")

X_train_seq = data["X_train_seq"]
L_train = data["L_train"]
X_test_seq = data["X_test_seq"]
L_test = data["L_test"]
train_order = data["train_order"]
test_order = data["test_order"]
train_pos = data["train_pos"]
test_pos = data["test_pos"]
y_aligned = data["y_aligned"]
dt_m_aligned = data["dt_m_aligned"]

## 6. PyTorch `Dataset` and `DataLoader`

A thin wrapper around the numpy arrays. We hand the Dataset whatever subset of indices
the current fold needs, so we don't have to copy big arrays.


In [ ]:
class WindowDataset(Dataset):
    def __init__(self, X, lengths, y=None):
        self.X = torch.from_numpy(X)
        self.lengths = torch.from_numpy(lengths.astype('int64'))
        self.y = None if y is None else torch.from_numpy(y.astype('float32'))

    def __len__(self):
        return self.X.shape[0]

    def __getitem__(self, i):
        if self.y is None:
            return self.X[i], self.lengths[i]
        return self.X[i], self.lengths[i], self.y[i]


def make_loader(X, lengths, y, batch_size, shuffle):
    return DataLoader(
        WindowDataset(X, lengths, y),
        batch_size=batch_size,
        shuffle=shuffle,
        num_workers=0,
        pin_memory=False,
        drop_last=False,
    )

## 7. PyTorch LSTM model

This is the structural mirror of arunmohan003's `SentimentRNN` — same `__init__` /
`forward` / `init_hidden` pattern — adapted for numeric input (no `nn.Embedding`).


In [ ]:
class FraudLSTM(nn.Module):
    """LSTM with masked mean+max pooling and an optional static-row tower."""
    def __init__(self, n_features, hidden_dim=HIDDEN_DIM, num_layers=NUM_LAYERS,
                 drop_prob=DROPOUT, output_dim=1, use_static_tower=USE_STATIC_TOWER):
        super().__init__()
        self.use_static_tower = use_static_tower

        self.lstm = nn.LSTM(
            input_size=n_features,
            hidden_size=hidden_dim,
            num_layers=num_layers,
            batch_first=True,
            dropout=drop_prob if num_layers > 1 else 0.0,
        )

        seq_out_dim = 2 * hidden_dim  # mean + max

        if use_static_tower:
            self.static_mlp = nn.Sequential(
                nn.Linear(n_features, 256), nn.ReLU(), nn.Dropout(drop_prob),
                nn.Linear(256, 128),        nn.ReLU(), nn.Dropout(drop_prob),
                nn.Linear(128, 64),         nn.ReLU(),
            )
            combined_dim = seq_out_dim + 64
        else:
            combined_dim = seq_out_dim

        self.head = nn.Sequential(
            nn.Linear(combined_dim, 64), nn.ReLU(), nn.Dropout(drop_prob),
            nn.Linear(64, output_dim),
        )

    def forward(self, x, lengths):
        # x: (B, T, F)  left-padded; lengths: (B,) real-step counts
        B, T, _ = x.shape
        lstm_out, _ = self.lstm(x)                                  # (B, T, H)

        # build a mask for the real (right-aligned) positions
        idx = torch.arange(T, device=x.device).unsqueeze(0)         # (1, T)
        pos_from_end = T - 1 - idx                                  # 0..T-1
        mask = pos_from_end < lengths.to(x.device).unsqueeze(1)     # (B, T)
        mask_f = mask.unsqueeze(-1).float()

        # masked mean
        sum_  = (lstm_out * mask_f).sum(dim=1)
        cnt   = mask_f.sum(dim=1).clamp(min=1.0)
        mean_pool = sum_ / cnt

        # masked max (pads → very negative)
        neg_inf  = torch.finfo(lstm_out.dtype).min
        max_pool = lstm_out.masked_fill(~mask.unsqueeze(-1), neg_inf).max(dim=1).values

        seq_vec = torch.cat([mean_pool, max_pool], dim=1)

        if self.use_static_tower:
            current  = x[:, -1, :]                                  # last real step
            stat_vec = self.static_mlp(current)
            feat = torch.cat([seq_vec, stat_vec], dim=1)
        else:
            feat = seq_vec

        return self.head(feat).squeeze(-1)

In [ ]:
# smoke test
N_FEATURES = X_train_seq.shape[2]
m = FraudLSTM(N_FEATURES).to(device)
xb = torch.randn(8, WINDOW, N_FEATURES, device=device)
lb = torch.randint(1, WINDOW+1, (8,), device=device)
print('forward smoke test out shape:', m(xb, lb).shape)
del m, xb, lb

In [ ]:
import torch

T = 6
lengths = torch.tensor([2, 4, 6])

idx = torch.arange(T).unsqueeze(0)
pos_from_end = T - 1 - idx

mask = pos_from_end < lengths.unsqueeze(1)
mask_f = mask.unsqueeze(-1).float()

print(idx)
print(pos_from_end)
print(lengths.unsqueeze(1))
print(mask)
print(mask_f)

In [ ]:
cnt   = mask_f.sum(dim=1).clamp(min=1.0)
print(cnt)

## 8. Expanding-window CV + training loop

Same fold scheme as the Keras notebook. Training loop mirrors arunmohan003's pattern:

1. Per epoch, iterate batches and reset hidden state for each batch (windows are
   independent).
2. `optimizer.zero_grad()` → `forward` → `loss.backward()` → `clip_grad_norm_` →
   `optimizer.step()`.
3. Validation pass with `model.eval()` and `torch.no_grad()`.
4. Early-stop on validation AUC, restore best weights.


In [ ]:
device = torch.device('cuda' if torch.cuda.is_available()
                       else 'mps' if torch.backends.mps.is_available()
                       else 'cpu')
print('device:', device)


def expanding_month_folds(months_array, min_train_months=MIN_TRAIN_MONTHS):
    months = sorted(np.unique(months_array).tolist())
    for vm in months[min_train_months:]:
        tm = [m for m in months if m < vm]
        ti = np.flatnonzero(np.isin(months_array, tm))
        vi = np.flatnonzero(months_array == vm)
        yield (vm, tm, ti, vi)

In [ ]:
def train_one_fold(X_tr, L_tr, y_tr, X_va, L_va, y_va, n_features,
                   epochs, batch, lr, weight_decay, device,
                   early_stop_patience, grad_clip, seed):
    torch.manual_seed(seed); np.random.seed(seed)
    model = FraudLSTM(n_features).to(device)

    if USE_POS_WEIGHT:
        pos = float((y_tr == 1).sum()); neg = float(len(y_tr) - pos)
        pw = torch.tensor([np.sqrt(neg / max(pos, 1.0))], device=device, dtype=torch.float32)
        loss_fn = nn.BCEWithLogitsLoss(pos_weight=pw)
    else:
        loss_fn = nn.BCEWithLogitsLoss()                            # plain BCE → better AUC

    optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=weight_decay)

    train_loader = make_loader(X_tr, L_tr, y_tr, batch_size=batch, shuffle=True)
    val_loader   = make_loader(X_va, L_va, y_va, batch_size=batch, shuffle=False)

    steps = max(1, math.ceil(len(X_tr) / batch))
    scheduler = torch.optim.lr_scheduler.OneCycleLR(
        optimizer, max_lr=lr, epochs=epochs, steps_per_epoch=steps,
        pct_start=0.1, anneal_strategy='cos',
    )

    best_auc, best_state, best_val_preds, bad = -1.0, None, None, 0
    for epoch in range(1, epochs + 1):
        model.train()
        t0 = time.time(); running, n_seen = 0.0, 0
        for xb, lb, yb in train_loader:
            xb = xb.to(device); lb = lb.to(device); yb = yb.to(device)
            optimizer.zero_grad()
            logits = model(xb, lb)
            loss = loss_fn(logits, yb)
            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), grad_clip)
            optimizer.step(); scheduler.step()
            running += loss.item() * xb.size(0); n_seen += xb.size(0)
        train_loss = running / max(n_seen, 1)

        model.eval(); preds = []
        with torch.no_grad():
            for xb, lb, _ in val_loader:
                xb = xb.to(device); lb = lb.to(device)
                preds.append(torch.sigmoid(model(xb, lb)).cpu().numpy())
        val_preds = np.concatenate(preds)
        val_auc = roc_auc_score(y_va, val_preds)

        print(f'   ep {epoch:>2}/{epochs}  loss={train_loss:.4f}  val_auc={val_auc:.4f}  ({time.time()-t0:.1f}s)')
        if val_auc > best_auc:
            best_auc = val_auc; best_state = copy.deepcopy(model.state_dict())
            best_val_preds = val_preds; bad = 0
        else:
            bad += 1
            if bad >= early_stop_patience:
                print(f'   early stop at epoch {epoch}'); break

    if best_state is not None: model.load_state_dict(best_state)
    return best_val_preds, best_auc, model

In [ ]:
# ===== run folds with seed ensembling =====
oof        = np.full(len(X_train_seq), np.nan, dtype=np.float32)
test_preds = np.zeros(len(X_test_seq), dtype=np.float32)
fold_aucs  = []

fold_specs = list(expanding_month_folds(dt_m_aligned, MIN_TRAIN_MONTHS))
print(f'{len(fold_specs)} folds; {N_SEEDS} seeds per fold')

for fold, (vm, tm, idxT, idxV) in enumerate(fold_specs):
    print(f'\n=== Fold {fold}: train {tm} → validate {vm} '
          f'(train={len(idxT):,}, valid={len(idxV):,}) ===')

    seed_val_preds, seed_test_preds = [], []
    for s in range(N_SEEDS):
        seed = SEED + s
        print(f'-- seed {seed} --')
        vp, va, model = train_one_fold(
            X_train_seq[idxT], L_train[idxT], y_aligned[idxT],
            X_train_seq[idxV], L_train[idxV], y_aligned[idxV],
            n_features=N_FEATURES, epochs=EPOCHS, batch=BATCH,
            lr=LR, weight_decay=WEIGHT_DECAY, device=device,
            early_stop_patience=EARLY_STOP_PATIENCE,
            grad_clip=GRAD_CLIP, seed=seed,
        )
        seed_val_preds.append(vp)

        # test preds from this seed's best model
        model.eval()
        test_loader = make_loader(X_test_seq, L_test, y=None, batch_size=BATCH, shuffle=False)
        tps = []
        with torch.no_grad():
            for xb, lb in test_loader:
                xb = xb.to(device); lb = lb.to(device)
                tps.append(torch.sigmoid(model(xb, lb)).cpu().numpy())
        seed_test_preds.append(np.concatenate(tps))

        del model
        if device.type == 'mps': torch.mps.empty_cache()
        gc.collect()

    fold_val   = np.mean(seed_val_preds, axis=0)
    fold_test  = np.mean(seed_test_preds, axis=0)
    fold_auc   = roc_auc_score(y_aligned[idxV], fold_val)
    print(f'   fold AUC (seed-avg) = {fold_auc:.4f}')
    fold_aucs.append((int(vm), float(fold_auc)))
    oof[idxV]  = fold_val
    test_preds += fold_test

if len(fold_specs):
    test_preds /= len(fold_specs)

validated = ~np.isnan(oof)
overall_auc = roc_auc_score(y_aligned[validated], oof[validated])
print(f'\n=== LSTM OOF AUC (validated months only) = {overall_auc:.4f} ===')
print(f'   per-fold: {fold_aucs}')

# APPLYING OPTUNA

In [ ]:
# import optuna
# from optuna.samplers import TPESampler
# from optuna.pruners import MedianPruner
# import copy
#
# def objective(trial: optuna.Trial) -> float:
#     """One Optuna trial = train all expanding-window folds with one config."""
#     # --- Search space ---
#     config = {
#         'LR':           trial.suggest_float('LR', 1e-5, 1e-3, log=True),
#         'HIDDEN_DIM':   trial.suggest_categorical('HIDDEN_DIM', [64, 128, 256]),
#         'NUM_LAYERS':   trial.suggest_categorical('NUM_LAYERS', [1, 2]),
#         'DROPOUT':      trial.suggest_float('DROPOUT', 0.1, 0.5, step=0.1),
#         # 'BATCH':        trial.suggest_categorical('BATCH', [256, 512, 1024]),
#         'BATCH':        BATCH,
#         'WEIGHT_DECAY': trial.suggest_float('WEIGHT_DECAY', 1e-7, 1e-3, log=True),
#         # 'GRAD_CLIP':    trial.suggest_float('GRAD_CLIP', 0.25, 2.0),
#         'GRAD_CLIP':    GRAD_CLIP,
#         # 'WINDOW':       trial.suggest_categorical('WINDOW', [3, 5, 10]),
#         'WINDOW':       WINDOW,
#     }
#
#     # If WINDOW changes, you must rebuild X_train_seq / X_test_seq for that trial.
#     # Cache by window size to avoid rebuilding repeatedly.
#     X_tr_seq, X_te_seq, y_al, dt_m_al = get_seq_for_window(config['WINDOW'])
#
#     # Build a temporary model class with the trial's hyperparameters
#     class TrialLSTM(nn.Module):
#         def __init__(self, n_features):
#             super().__init__()
#             self.hidden_dim = config['HIDDEN_DIM']
#             self.num_layers = config['NUM_LAYERS']
#             self.lstm = nn.LSTM(input_size=n_features,
#                                 hidden_size=config['HIDDEN_DIM'],
#                                 num_layers=config['NUM_LAYERS'],
#                                 batch_first=True, dropout=0.0)
#             self.fc1 = nn.Linear(config['HIDDEN_DIM'], 32)
#             self.relu = nn.ReLU()
#             self.dropout = nn.Dropout(config['DROPOUT'])
#             self.fc2 = nn.Linear(32, 1)
#         def forward(self, x, hidden):
#             out, hidden = self.lstm(x, hidden)
#             out = self.dropout(out[:, -1, :])
#             out = self.relu(self.fc1(out))
#             out = self.dropout(out)
#             return self.fc2(out).squeeze(-1), hidden
#         def init_hidden(self, bs, dev):
#             return (torch.zeros(self.num_layers, bs, self.hidden_dim, device=dev),
#                     torch.zeros(self.num_layers, bs, self.hidden_dim, device=dev))
#
#     # Run all expanding-window folds, collect AUCs
#     fold_aucs = []
#     for fold, (vm, tm, idxT, idxV) in enumerate(
#             expanding_month_folds(dt_m_al, MIN_TRAIN_MONTHS)):
#
#         model = TrialLSTM(X_tr_seq.shape[2]).to(device)
#         pos = (y_al[idxT] == 1).sum(); neg = len(idxT) - pos
#         pos_w = torch.tensor([np.sqrt(neg / max(pos, 1.0))], device=device, dtype=torch.float32)
#         loss_fn = nn.BCEWithLogitsLoss(pos_weight=pos_w)
#         opt = torch.optim.AdamW(model.parameters(), lr=config['LR'],
#                                 weight_decay=config['WEIGHT_DECAY'])
#
#         train_loader = make_loader(X_tr_seq[idxT], y_al[idxT], config['BATCH'], shuffle=True)
#         val_loader   = make_loader(X_tr_seq[idxV], y_al[idxV], config['BATCH'], shuffle=False)
#
#         best_auc, bad = -1.0, 0
#         for epoch in range(EPOCHS):
#             model.train()
#             for xb, yb in train_loader:
#                 xb, yb = xb.to(device), yb.to(device)
#                 h = model.init_hidden(xb.size(0), device)
#                 opt.zero_grad()
#                 logits, _ = model(xb, h)
#                 loss = loss_fn(logits, yb)
#                 loss.backward()
#                 nn.utils.clip_grad_norm_(model.parameters(), config['GRAD_CLIP'])
#                 opt.step()
#
#             model.eval()
#             preds = []
#             with torch.no_grad():
#                 for xb, yb in val_loader:
#                     h = model.init_hidden(xb.size(0), device)
#                     logits, _ = model(xb.to(device), h)
#                     preds.append(torch.sigmoid(logits).cpu().numpy())
#             val_auc = roc_auc_score(y_al[idxV], np.concatenate(preds))
#
#             if val_auc > best_auc:
#                 best_auc, bad = val_auc, 0
#             else:
#                 bad += 1
#                 if bad >= EARLY_STOP_PATIENCE: break
#
#         fold_aucs.append(best_auc)
#
#         # === Pruning — abandon a bad trial early ===
#         trial.report(np.mean(fold_aucs), fold)
#         if trial.should_prune():
#             raise optuna.TrialPruned()
#
#     return float(np.mean(fold_aucs))
#
#
# # Cache windowed arrays per WINDOW value so we don't rebuild each trial
# _seq_cache = {}
# def get_seq_for_window(window):
#     if window not in _seq_cache:
#         Xall_seq, order = build_uid_windows(X_all, feature_cols, window)
#         src = X_all.loc[order, '__source__'].to_numpy()
#         Xtr  = Xall_seq[src == 'train']
#         Xte  = Xall_seq[src == 'test']
#         ytr  = y_train.loc[order[src == 'train']].to_numpy(np.float32)
#         dtm  = X_all.loc[order[src == 'train'], 'DT_M'].to_numpy()
#         _seq_cache[window] = (Xtr, Xte, ytr, dtm)
#     return _seq_cache[window]
#
#
# # === Run the study ===
# sampler = TPESampler(seed=SEED)
# pruner  = MedianPruner(n_startup_trials=5, n_warmup_steps=1)
# study = optuna.create_study(direction='maximize', sampler=sampler, pruner=pruner,
#                             study_name='lstm_fraud', storage=None)
#
# study.optimize(objective, n_trials=30, show_progress_bar=True)
#
# print('best AUC :', study.best_value)
# print('best params:')
# for k, v in study.best_params.items():
#     print(f'  {k:14s} = {v}')

In [ ]:
# print('Completed trials :', len([t for t in study.trials if t.state.name == 'COMPLETE']))
# print('Pruned trials    :', len([t for t in study.trials if t.state.name == 'PRUNED']))
# print('Best AUC         :', study.best_value)
# print('Best params      :')
# for k, v in study.best_params.items():
#     print(f'  {k:14s} = {v}')

In [ ]:
# import importlib.util
# print(importlib.util.find_spec("plotly"))

In [ ]:
# import importlib
# import optuna.visualization
# importlib.reload(optuna.visualization)
#
# # Which hyperparameter mattered most for AUC?
# optuna.visualization.plot_optimization_history(study).show()
# optuna.visualization.plot_param_importances(study).show()  # tells you which hyperparams mattered

In [ ]:
# best = study.best_params
# print(best)
# # Re-create FraudLSTM and train_one_fold with these values

## 9. Save OOF and test predictions

Same format as the Keras notebook so you can mix-and-match for the comparison.


In [ ]:
# oof_df = pd.DataFrame({'TransactionID': train_order, 'oof_lstm_torch': oof})
# oof_df = oof_df.set_index('TransactionID').reindex(X_train.index).reset_index()
# oof_df.to_csv('oof_lstm_torch.csv', index=False)
#
# test_df = pd.DataFrame({'TransactionID': test_order, 'pred_lstm_torch': test_preds})
# test_df = test_df.set_index('TransactionID').reindex(X_test.index).reset_index()
# test_df.to_csv('test_pred_lstm_torch.csv', index=False)
#
# print('Wrote oof_lstm_torch.csv and test_pred_lstm_torch.csv')


## 10. Notes — what's the same vs the Keras version, and the sentiment reference

**Same as Keras version:**

- Bridge train+test, gap features, scaling
- Per-UID windowing
- Expanding-window CV
- File outputs (different filenames so they don't collide)

**Differences from Keras → PyTorch:**

| Topic | Keras | PyTorch (this notebook) |
|--|--|--|
| Training loop | `model.fit(...)` does it implicitly | Hand-written loop (mirrors arunmohan003) |
| Loss | `binary_crossentropy` + `class_weight` | `BCELoss(reduction='none')` + per-sample weight |
| Hidden state | implicit | explicit `init_hidden(batch_size, device)` per batch |
| Class imbalance | `class_weight={0:1, 1:w}` arg | `pos_weight = neg/pos` applied per sample |
| Best-model retention | `restore_best_weights=True` callback | `copy.deepcopy(state_dict)` after each epoch |
| Best-epoch criterion | `val_auc` via Keras callback | `val_auc` via the loop (same idea) |
| Device handling | TF auto-discovers Metal | explicit `device = mps/cuda/cpu` |

**Differences from arunmohan003's sentiment notebook:**

| Topic | Sentiment | Fraud (this notebook) |
|--|--|--|
| Input type | word indices | numeric features |
| `nn.Embedding` | yes | **no** — features are already numeric |
| Sequence length | up to 500 (review length) | 5 (small UID window) |
| Best-model criterion | val loss | val AUC |
| Class imbalance | balanced (50/50 IMDB) | very imbalanced (~3.5% fraud) — needs `pos_weight` |
| Output domain | sentiment 0/1 | fraud 0/1 (same shape, different meaning) |

**Tuning knobs to try after the baseline runs:**

- `WINDOW` — try 3, 10, 20
- `HIDDEN_DIM` — try 64, 256
- `NUM_LAYERS` — try 1 (no input dropout)
- `DROPOUT` — try 0.2, 0.5
- `BATCH` — try 2048 or 4096 if MPS memory allows
- Replace `LSTM` with `GRU` (one-line swap in `__init__`)

For the methodology chapter, comparing the AUC from this PyTorch notebook with the
AUC from the Keras notebook (both under expanding-window CV) is also instructive —
they should be close but not identical due to optimizer / weight-init differences.
